In [1]:
import json, os, sys
sys.path.append("../")
import pandas as pd
import numpy as np
from steps.utils.generic import read_json_or_jsonl
from tqdm import tqdm

In [2]:
QA_METADATA_FOLDER = "/scratch/lamdo/IRB/qa_metadata/2Dec2025 (main)/"
RETRIEVAL_METADATA_FOLDER = "/scratch/lamdo/IRB/retrieval_metadata/2Dec2025/"
QRELS_PATH = "/scratch/lamdo/IRB/benchmarks/2Dec2025/irb/qrels/test.tsv"

In [3]:
def read_qrels(qrel_path):
    _qrels = pd.read_csv(qrel_path, sep='\t').to_dict("records")
    
    metadata = {}
    for line in _qrels:
        query_id = str(line["query-id"])
        doc_id = str(line["corpus-id"])
        score = line["score"]

        if query_id not in metadata:
            metadata[query_id] = {}

        metadata[query_id][doc_id] = int(score)

    return metadata

    
def get_retrieval_status(qrels, retrieval_metadata, num_retrieval_contexts = 5):
    queries_ids = list(retrieval_metadata["full"].keys())

    res = {}
    for query_id in queries_ids:
        groundtruth_urls = qrels[query_id].keys()
        prediction_urls = [line["docid"] for line in retrieval_metadata["full"][query_id]][:num_retrieval_contexts]
        retrieval_status = all([gtu in prediction_urls for gtu in groundtruth_urls])
        
        res[query_id] = retrieval_status

    return res


def get_closed_book_status(closed_book_qa_eval_results):
    queries_ids = list(closed_book_qa_eval_results.keys())

    res = {}
    for query_id in queries_ids:
        eval_res = closed_book_qa_eval_results[query_id]
        if eval_res.get("CORRECT", 0) >= 0.999:
            res[query_id] = True

        else: res[query_id] = False

    return res

def get_average_performance(eval_res_list, split: str):
    correct, incorrect, not_attempted = [], [], []
    truthfulness = []

    for corr, incorr, not_att in eval_res_list:
        correct.append(corr)
        incorrect.append(incorr)
        not_attempted.append(not_att)


        truthfulness.append(corr - incorr)
    
    return {
        "split": split,
        "support": len(eval_res_list),
        "correct": round(np.mean(correct), 3),
        "incorrect": round(np.mean(incorrect), 3),
        "not_attempted": round(np.mean(not_attempted), 3),
        "truthfulness": round(np.mean(truthfulness), 3)
    }


def get_splits(qrels, 
                retrieval_metadata, 
                closed_book_qa_eval_results, 
                rag_qa_eval_results, 
                num_retrieval_contexts = 5,
                premise = True):
    retrieval_status = get_retrieval_status(qrels, retrieval_metadata, num_retrieval_contexts)
    closed_book_status = get_closed_book_status(closed_book_qa_eval_results)

    all_performance = {}

    ids = {
        "redundant": [query_id for query_id in retrieval_status if retrieval_status[query_id] and closed_book_status[query_id]],
        "resilience": [query_id for query_id in retrieval_status if not retrieval_status[query_id] and closed_book_status[query_id]],
        "augmentation": [query_id for query_id in retrieval_status if retrieval_status[query_id] and not closed_book_status[query_id]],
        "hopeless": [query_id for query_id in retrieval_status if not retrieval_status[query_id] and not closed_book_status[query_id]]
    }

    for mode in ids:
        eval_results = []
        mode_ids = ids[mode]
        for query_id in mode_ids:
            if premise and query_id.startswith("~"): continue
            elif not premise and not query_id.startswith("~"): continue
            
            eval_res = rag_qa_eval_results[query_id]
            corr, incorr, not_att = float(eval_res.get("CORRECT", 0)), float(eval_res.get("INCORRECT", 0)), float(eval_res.get("NOT_ATTEMPTED", 0))
            
            eval_results.append([corr, incorr, not_att])
        all_performance[mode] = get_average_performance(eval_results, split = mode)

    return all_performance



In [4]:
metric = "correct"

data = []
for model_name in tqdm(["gpt-4.1-mini", "gpt-4.1", "gpt-5-mini", "gpt-5", "llama-4-scout", "llama-3.3-70B", "gpt-oss-120b", "deepseek-r1"]):
    qrels = read_qrels(QRELS_PATH)
    retrieval_metadata = read_json_or_jsonl(os.path.join(RETRIEVAL_METADATA_FOLDER, "irb__text-embedding-3-small.json"))
    rag_qa_eval_results = read_json_or_jsonl(os.path.join(QA_METADATA_FOLDER, f"irb__{model_name}__text-embedding-3-small__chunk0", "eval_result.json"))
    closed_book_qa_eval_results = read_json_or_jsonl(os.path.join(QA_METADATA_FOLDER, f"irb__{model_name}", "eval_result.json"))
    
    perf = get_splits(qrels, retrieval_metadata, closed_book_qa_eval_results, rag_qa_eval_results, premise = False)

    to_append = {
        "model_name": model_name,
        **{mode: round(perf[mode][metric] * 100, 1) for mode in ["redundant", "resilience", "augmentation", "hopeless"]}
    }
    data.append(to_append)

df = pd.DataFrame(data)
df.to_csv("interplay_correcness.csv", index = False)

100%|██████████| 8/8 [01:10<00:00,  8.78s/it]
